In [ ]:
import sys
import os
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models, optimizers
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
sys.path.append(os.path.abspath(".."))
import config

In [ ]:
train_dir = "../dataset_train/"
test_dir = "../dataset_test/"

img_width, img_height = config.IMG_SIZE
batch_size = 32

train_data_fraction = 0.1  
test_data_fraction = 0.1  


In [ ]:
datagen = ImageDataGenerator(rescale=1./255)

train_generator = datagen.flow_from_directory(
    train_dir,
    target_size=(img_width, img_height),
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=True  
)

validation_generator = datagen.flow_from_directory(
    test_dir,
    target_size=(img_width, img_height),
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=True
)

In [ ]:
train_steps = int(train_generator.samples * train_data_fraction / batch_size)
validation_steps = int(validation_generator.samples *
                       test_data_fraction / batch_size)

print(f"Using {train_steps * batch_size} training images and {validation_steps * batch_size} testing images.")

In [ ]:
model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu',
                  input_shape=(img_width, img_height, 3)),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(train_generator.num_classes, activation='softmax')
])


model.compile(optimizer=optimizers.Adam(),
              loss='categorical_crossentropy',
              metrics=['accuracy'])


model.summary()

In [ ]:
epochs = 10
history = model.fit(
    train_generator,
    steps_per_epoch=train_steps,
    epochs=epochs,
    validation_data=validation_generator,
    validation_steps=validation_steps
)

In [ ]:

loss, accuracy = model.evaluate(validation_generator, steps=validation_steps)
print(f"Test Loss: {loss:.4f}, Test Accuracy: {accuracy:.4f}")